# Student Performance Analysis Using Data Analytics and AI
**Prepared by:** Lohitha Galeti  
**Project type:** Exploratory Data Analysis (EDA) and introductory machine learning

## 1. Problem statement
Educational stakeholders may benefit from understanding how students' study habits, previous grades, absences, and other factors relate to final academic results. This project explores a public student-performance dataset, summarizes patterns with visualizations, and demonstrates a simple regression model.

## 2. Objectives
- Load and inspect the dataset.
- Check data quality and summarize key variables.
- Visualize selected relationships with final grades.
- Train a simple baseline model to estimate the final grade from selected features.
- Interpret results cautiously and identify limitations.

## 3. Dataset
The primary dataset is the **Student Performance** dataset from the UCI Machine Learning Repository. It contains student achievement and background attributes for Portuguese secondary-school students. The notebook attempts to download the official dataset automatically. If the download is unavailable, it creates a clearly labeled synthetic demonstration dataset so that the workflow can still be explored; results from synthetic data must not be presented as findings from the UCI dataset.

Source: https://archive.ics.uci.edu/dataset/320/student+performance


In [ ]:
import io
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

## 4. Load the data
The official UCI archive is attempted first. If internet access is unavailable, the notebook uses synthetic data for demonstration and labels it clearly.

In [ ]:
DATA_SOURCE = "UCI Student Performance dataset"
used_synthetic_fallback = False

try:
    url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
    with urllib.request.urlopen(url, timeout=20) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as zf:
        csv_name = next(name for name in zf.namelist() if name.endswith("student-mat.csv"))
        with zf.open(csv_name) as file:
            df = pd.read_csv(file, sep=";")
    print("Loaded the official UCI mathematics student dataset.")
except Exception as error:
    used_synthetic_fallback = True
    DATA_SOURCE = "Synthetic demonstration data (official UCI download unavailable)"
    rng = np.random.default_rng(42)
    n = 200
    studytime = rng.integers(1, 5, size=n)
    absences = rng.integers(0, 16, size=n)
    G1 = np.clip(np.round(rng.normal(10 + studytime * 1.5, 3, size=n)), 0, 20).astype(int)
    G2 = np.clip(np.round(G1 + rng.normal(1, 2.5, size=n)), 0, 20).astype(int)
    G3 = np.clip(np.round(0.25 * G1 + 0.65 * G2 + 1.0 * studytime - 0.12 * absences
                          + rng.normal(0, 2, size=n)), 0, 20).astype(int)
    df = pd.DataFrame({
        "studytime": studytime, "absences": absences, "G1": G1, "G2": G2, "G3": G3
    })
    print("Could not download the official dataset. Using synthetic data for demonstration only.")
    print("Download error:", str(error)[:200])

print("Data source:", DATA_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

## 5. Understand and clean the data
Review columns, data types, missing values, duplicate rows, and descriptive statistics. The notebook keeps the analysis simple and transparent.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

duplicate_count = int(df.duplicated().sum())
print("Duplicate rows:", duplicate_count)

# Remove exact duplicate rows and make a working copy.
df = df.drop_duplicates().copy()

print("\\nDescriptive statistics:")
display(df.describe(include="all").T)

## 6. Exploratory data analysis (EDA)
The charts below summarize final grades and explore selected relationships. The grade columns G1, G2, and G3 represent first-period, second-period, and final grades in the UCI dataset.

In [ ]:
if "G3" not in df.columns:
    raise ValueError("Expected target column 'G3' was not found.")

plt.figure(figsize=(8, 5))
plt.hist(df["G3"].dropna(), bins=11, edgecolor="black")
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Final grade (0–20)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

if "studytime" in df.columns:
    avg_grade = df.groupby("studytime", dropna=False)["G3"].mean()
    plt.figure(figsize=(8, 5))
    avg_grade.plot(kind="bar", edgecolor="black")
    plt.title("Average Final Grade by Weekly Study-Time Category")
    plt.xlabel("Study-time category (higher values indicate more study time)")
    plt.ylabel("Average final grade (G3)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

if "absences" in df.columns:
    plt.figure(figsize=(8, 5))
    plt.scatter(df["absences"], df["G3"], alpha=0.7)
    plt.title("Absences vs Final Grade")
    plt.xlabel("Number of absences")
    plt.ylabel("Final grade (G3)")
    plt.tight_layout()
    plt.show()

grade_columns = [c for c in ["G1", "G2", "G3"] if c in df.columns]
if len(grade_columns) >= 2:
    print("Grade correlations:")
    display(df[grade_columns].corr().round(3))
    plt.figure(figsize=(6, 5))
    plt.imshow(df[grade_columns].corr(), aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(grade_columns)), grade_columns)
    plt.yticks(range(len(grade_columns)), grade_columns)
    plt.title("Correlation Between Grade Periods")
    plt.tight_layout()
    plt.show()

## 7. Introductory machine-learning demonstration
A linear regression baseline estimates final grade using available columns among G1, G2, studytime, and absences. The model is for educational demonstration, not for high-stakes decisions about students. Grades from earlier periods may strongly predict final grades, so this is not an independent causal explanation.

In [ ]:
candidate_features = ["G1", "G2", "studytime", "absences"]
features = [c for c in candidate_features if c in df.columns]
model_df = df[features + ["G3"]].dropna()

if len(features) >= 1 and len(model_df) >= 10:
    X = model_df[features]
    y = model_df["G3"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))
    r2 = r2_score(y_test, predictions)

    print("Features used:", features)
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared (R²): {r2:.3f}")

    comparison = pd.DataFrame({
        "Actual final grade": y_test.to_numpy(),
        "Predicted final grade": np.round(predictions, 2)
    })
    display(comparison.head(10))

    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, predictions, alpha=0.7)
    plt.plot([0, 20], [0, 20], linestyle="--")
    plt.xlim(0, 20)
    plt.ylim(0, 20)
    plt.title("Actual vs Predicted Final Grades")
    plt.xlabel("Actual grade")
    plt.ylabel("Predicted grade")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough usable rows or features to train the demonstration model.")

## 8. Key findings and interpretation
Use the charts and printed metrics generated above to write your final observations. Do not invent findings: describe what your run actually shows.

Suggested questions:
1. What does the distribution of final grades look like?
2. How do average grades vary across study-time categories in this dataset?
3. Does the scatter plot show a clear, weak, or unclear relationship between absences and final grades?
4. How close are the model's predictions to the actual grades according to MAE and RMSE?
5. What are two limitations of the analysis?

## 9. Limitations and responsible use
- Correlation does not establish causation.
- The dataset reflects a particular context and should not be generalized to all schools or students.
- A simple train/test split and linear regression are only a baseline; model performance may vary.
- Grades from earlier periods may be closely related to final grades and can make prediction easier.
- Predictions should not be used to label, punish, or make high-stakes decisions about individual students.
- If synthetic fallback data was used, the charts and metrics are demonstrations only and are not empirical findings.

## 10. Conclusion
This project demonstrates a basic end-to-end analytics workflow: data loading, data-quality checks, exploratory analysis, visualization, and an introductory predictive model. The results should be interpreted in the context of the actual dataset used. Further work could include cross-validation, comparing models, and consulting educators before drawing practical recommendations.

## References
- UCI Machine Learning Repository, Student Performance dataset: https://archive.ics.uci.edu/dataset/320/student+performance
- pandas documentation: https://pandas.pydata.org/docs/
- scikit-learn documentation: https://scikit-learn.org/stable/
